# Simulate Cost Matrix

Run the cells in order. This notebook stores the raw simulated data and the expected-cost summary for each uncertainty scenario.

In [ ]:
import os
import random

import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import minimize_scalar
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
# Parameters
ITERATIONS = 1000
SHORTFALL_COST = 2.0  # c^s
UNCERTAINTY_LEVELS = {
    "none": 0.0,
    "low": 0.25,
    "med": 0.5,
    "high": 0.75,
}
SEED = 67

levels = list(UNCERTAINTY_LEVELS.keys())
total_cost_matrix = [[0.0 for _ in levels] for _ in levels]
estimate_samples_matrix = [[None for _ in levels] for _ in levels]

# Raw and summary storage
scenario_data = {}
summary_rows = []

os.makedirs("Figures", exist_ok=True)

In [ ]:
# Run the simulation, optimize S, and store the results
header = (
    f"{'Demand':<6} {'Estimate':<8} "
    f"{'Exp inv cost':>14} {'Exp short cost':>16} {'Exp total cost':>16}"
)
print(header)
print("-" * len(header))

scenario_data.clear()
summary_rows.clear()
total_cost_matrix = [[0.0 for _ in levels] for _ in levels]
estimate_samples_matrix = [[None for _ in levels] for _ in levels]

for demand_label, demand_sigma in UNCERTAINTY_LEVELS.items():
    for estimate_label, estimate_sigma in UNCERTAINTY_LEVELS.items():
        rng = random.Random(f"{SEED}:{demand_label}:{estimate_label}")

        D = []
        E = []

        for _ in range(ITERATIONS):
            if demand_sigma == 0:
                demand = 1.0
            else:
                demand_mu = -0.5 * demand_sigma * demand_sigma
                demand = rng.lognormvariate(demand_mu, demand_sigma)

            if estimate_sigma == 0:
                estimate_noise = 1.0
            else:
                estimate_mu = -0.5 * estimate_sigma * estimate_sigma
                estimate_noise = rng.lognormvariate(estimate_mu, estimate_sigma)

            estimate = demand * estimate_noise

            D.append(demand)
            E.append(estimate)

        result = minimize_scalar(
            lambda S: S + SHORTFALL_COST * sum(max(e - S, 0.0) for e in E) / ITERATIONS,
            bounds=(0.0, max(E)),
            method="bounded",
        )

        S = result.x
        expected_investment_cost = S
        expected_shortfall_cost = SHORTFALL_COST * sum(max(e - S, 0.0) for e in E) / ITERATIONS
        expected_total_cost = expected_investment_cost + expected_shortfall_cost

        row_index = levels.index(demand_label)
        col_index = levels.index(estimate_label)
        total_cost_matrix[row_index][col_index] = expected_total_cost
        estimate_samples_matrix[row_index][col_index] = E.copy()

        scenario_data[(demand_label, estimate_label)] = {
            "D": D.copy(),
            "E": E.copy(),
            "expected_investment_cost": expected_investment_cost,
            "expected_shortfall_cost": expected_shortfall_cost,
            "expected_total_cost": expected_total_cost,
            "optimal_S": S,
        }

        summary_rows.append({
            "demand_level": demand_label,
            "estimate_level": estimate_label,
            "expected_investment_cost": expected_investment_cost,
            "expected_shortfall_cost": expected_shortfall_cost,
            "expected_total_cost": expected_total_cost,
            "optimal_S": S,
        })

        print(
            f"{demand_label:<6} "
            f"{estimate_label:<8} "
            f"{expected_investment_cost:>14.4f} "
            f"{expected_shortfall_cost:>16.4f} "
            f"{expected_total_cost:>16.4f}"
        )

In [ ]:
# Build a table you can sort, filter, and inspect
scenario_table = pd.DataFrame(summary_rows)
scenario_table

In [ ]:
# Example: inspect the raw D and E values for one scenario
scenario_data[("low", "high")]["D"][:10], scenario_data[("low", "high")]["E"][:10]

In [ ]:
# Create and save the matrix plot
cmap = LinearSegmentedColormap.from_list("white_to_blue", ["#ffffff", "#6baed6"])

fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(total_cost_matrix, origin="lower", cmap=cmap)

ax.set_xticks(range(len(levels)))
ax.set_yticks(range(len(levels)))
ax.set_xticklabels(levels)
ax.set_yticklabels(levels)
ax.set_xlabel("Estimate uncertainty")
ax.set_ylabel("Demand uncertainty")
ax.set_title("Expected total cost by uncertainty scenario")

for row_index in range(len(levels)):
    for col_index in range(len(levels)):
        value = total_cost_matrix[row_index][col_index]
        ax.text(col_index, row_index, f"{value:.2f}", ha="center", va="center")

colorbar = plt.colorbar(image, ax=ax)
colorbar.set_label("Expected total cost")

plt.tight_layout()
plt.savefig("Figures/expected_total_cost_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Create and save a matrix of histograms for the simulated E values
all_E_values = []
for row in estimate_samples_matrix:
    for samples in row:
        all_E_values.extend(samples)

global_max_E = max(all_E_values)

fig, axes = plt.subplots(len(levels), len(levels), figsize=(12, 10), sharex=True, sharey=True)

for row_index in range(len(levels)):
    for col_index in range(len(levels)):
        plot_row = len(levels) - 1 - row_index
        ax = axes[plot_row][col_index]
        E_values = estimate_samples_matrix[row_index][col_index]

        ax.hist(
            E_values,
            bins=25,
            range=(0.0, global_max_E),
            color="#9ecae1",
            edgecolor="#3182bd",
        )
        ax.set_title(f"{levels[row_index]}/{levels[col_index]}", fontsize=10)

        if plot_row == len(levels) - 1:
            ax.set_xlabel("E")
        if col_index == 0:
            ax.set_ylabel("Count")

fig.suptitle("Distribution of E by uncertainty scenario", fontsize=14)
plt.tight_layout(rect=(0, 0, 1, 0.97))
plt.savefig("Figures/estimate_histogram_matrix.png", dpi=300, bbox_inches="tight")
plt.show()